1. Prepare all nodes:
  + all nodes can communicate with each other and installed ubuntu
  + run the commands on all nodes
    ```bash                                                    
     sudo apt update && sudo apt upgrade -y                               
     sudo apt install -y curl wget vim gnupg2 software-properties-common apt-transport-https ca-certificates
    ``` 
  + disable swap (required by K8s)                                                       
    ```bash
     sudo swapoff -a # this disables swap for the current session, but it will come back when rebooting the system                              sudo sed -i '/ swap / s/^/#/' /etc/fsta # find rows having the word 'swap' and add # at the begining of row in /etc/fsta file 
    ``` 
  + open /etc/fstab, comment out the line starts with /swap.img  
  + install RKE2
    ```bash
      curl -sfL https://get.rke2.io | sudo sh - # download script from rke2, feed the script to sudo sh and execute from stdin
    ```  

2. Configure Master nodes
  + on master node 1 run (select node 1 as master node)
    ```bash
     sudo mkdir -p /etc/rancher/rke2
     sudo vi /etc/rancher/rke2/config.yaml
    ``` 
  + the config.yaml file on node 1 contains:
        + tls-san: tls subject alternative names. This tells K8s which hostnames or IP addresses should be included in the Kubernetes API server’s TLS certificate
        + we need to list all the master node's ip addresses   
    ```YAML
    tls-san:
      - 10.10.10.5
      - 10.10.10.35      
    etcd: true  
    ```
  + start rke2 service as server by
    `sudo systemctl enable rke2-server --now`
    + this is the same as running `systemctl enable and systemctl start` together  
  + from the file using `sudo cat /var/lib/rancher/rke2/server/node-token`, copy the content of the file as the token for this cluster
  + on master node 2, run `sudo mkdir -p /etc/rancher/rke2` to create the rke2 config file, and then edit the config.yaml file
  + edit the config.yaml file on node 2 as the following:
    ```YAML
    token: <token_value>
    tls-san:
      - 10.10.10.5
      - 10.10.10.35
    server: https://10.10.10.5:9345 # only need to include node 1's ip address for other nodes to join the cluster using node 1's token
    etcd: true
    ``````
 enable and run the rke2-server system service by running                  
        `sudo systemctl enable rke2-server --now`
  + add RKE2's binary directory to server node's PATH.
    + This ensure you can run kubctl, crictl and other rke2 commands without listing the full path of commands
        ```bash
        echo 'export PATH=$PATH:/var/lib/rancher/rke2/bin' >> ~/.bashrc
        source ~/.bashrc  
        ```
  + test if you can access kubectl by `kubectl get nodes` and make sure you can see both master nodesnodes
3. set up worker nodes for node 3 (worker node + etcd)
  + process step 1.
  + on node 3                        
    ```bash
     sudo mkdir -p /etc/rancher/rke2                                
     sudo vi /etc/rancher/rke2/config.yaml
    ``` 
  + the config.yaml file on node 3 contains:                                           
    ```YAML                    
    token: <your-cluster-token>                    
    server: https:10.10.10.5:9345                        
    etcd: true                      
    disable:                          
      - rke2-controller-manager               
      - rke2-scheduler              
      - rke2-cloud-controller-manager            
      - rke2-cordns                
      - rke2-ingress-nginx                 
      - rke2-metrics-server                
    ```
  + run    
    ```bash
    sudo systemctl enable rke2-server
    sudo systemctl start rke2-server
    ```
4. set up node 4 as a worker node
  + process step 1.
  + on node 4, run
    ```bash
     sudo mkdir -p /etc/rancher/rke2
     sudo vi /etc/rancher/rke2/config.yaml
    ``` 
  + the config.yaml file on 
    ```YAML
    token: <cluster-token>
    server: https:10.10.10.5:9345
    ```
  + run the following shell commands:    
    ```bash
    sudo systemctl enable rke2-server
    sudo systemctl start rke2-server
    ```



### Set up Airflow on K8s cluster using helm
1. set up helm on one of the master node
  ```shell
  sudo apt update
  sudo apt install -y apt-transport-https curl
  curl https://baltocdn.com/helm/signing.asc | sudo apt-key add -  echo "deb https://baltocdn.com/helm/stable/debian/ all main" | sudo tee /etc/apt/sources.list.d/helm-stable-debian.list
  sudo apt update  
sudo apt install -y helm
  # check helm version
  helm version
  ```
2. add helm to repo
   ``` shell
   helm repo add apache-airflow https://airflow.apache.org   
   helm repo update
   ```
3. create an airflow namespace
  `kubectl create namespace airflow`


### set up ingress controller on RKE2 cluster
* the helm command for installation:
    ```bash
    helm install custom-nginx ingress-nginx/ingress-nginx \
      --namespace ingress-nginx --create-namespace \
      --set controller.ingressClass=custom-nginx \
      --set controller.ingressClassResource.name=custom-nginx \
      --set controller.service.type=NodePort \
      --set controller.service.nodePorts.http=30080 \
      --set controller.service.nodePorts.https=30443
    ```

* if successfully installed, you will see the following outputs:
  
  The ingress-nginx controllerhas been installed.                   
   Get the application URL by running these commands:
  ```bash
    export HTTP_NODE_PORT=30080
    export HTTPS_NODE_PORT=30443
    export NODE_IP="$(kubectl get nodes --output jsonpath="{.items[0].status.addresses[1].address}")"
     
    echo "Visit http://${NODE_IP}:${HTTP_NODE_PORT} to access your application via HTTP."
    echo "Visit https://${NODE_IP}:${HTTPS_NODE_PORT} to access your application via HTTPS."
  ```

* An example Ingress that makes use of the controller:
  ```YAML
  apiVersion: networking.k8s.io/v1
  kind: Ingress
  metadata:
    name: example
    namespace: foo
  spec:
    ingressClassName: custom-nginx
    rules:
      - host: www.example.com
        http:
          paths:
            - pathType: Prefix
              backend:
                service:
                  name: exampleService
                  port:
                    number: 80
              path: /
    # This section is only required if TLS is to be enabled for the Ingress
    tls:
      - hosts:
        - www.example.com
        secretName: example-tls
  ```

 * If TLS is enabled for the Ingress, a Secret containing the certificate and key must also be provided:
    ```YAML
      apiVersion: v1
      kind: Secret
      metadata:
        name: example-tls
        namespace: foo
      data:
        tls.crt: <base64 encoded cert>
        tls.key: <base64 encoded key>
      type: kubernetes.io/tls
    ```

### install docker engine on Ubuntu server   
```bash  
  sudo apt-get update
  sudo apt-get install -y ca-certificates curl gnupg lsb-releas

  sudo mkdir -p /etc/apt/keyrings
  curl -fsSL https://download.docker.com/linux/ubuntu/gpg | \
  sudo gpg --dearmor -o /etc/apt/keyrings/docker.gpg    
  echo 
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.gpg]
  https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable" \
  sudo tee /tec/apt/sources.list.d/docker.list > /dev/null

  sudo apt-get update
  sudo apt-get install -y docker-ce docker-ce-cli docker-buildx-plugin docker-compose-plugin containerd.io
  sudo docker run hello-world
  sudo usermod -aG docker $USER # add the current user to the newly creared docker group so you can run docker commands without sudo
  newgrp docker # refresh to update the docker group with user membership
``` 